<h2> Predicting Biological Age from DNA Methylation Biomarkers </h2>

<h4>This notebook began as an assignment for my Data Science class, predicting biological age from DNA methylation (CpG) biomarkers. I extended it beyond the original scope with a comparison against Horvath's published 353-CpG epigenetic clock, along with additional analysis and cleanup for use as a standalone project. The analysis covers data cleaning, feature selection under extreme dimensionality, linear and polynomial regression modeling, and an evaluation of the selected features against real, published clock data. Full results and discussion are available in the README.</h4>

<h3>Read the data into a dataframe and inspect the dataset</h3>

In [60]:
import pandas as pd
import numpy as np
import biolearn
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score

# Dataset file url.
url = "https://gitlab.cs.man.ac.uk/13212/lab5-dataset/-/raw/e4c3603dcbf707eff604c86274dd3bb52796dc0d/age_regression_data.csv"
df = pd.read_csv(url)
df.head()

/var/folders/dt/gxvjykvd0qs2brpp2xqft8c40000gn/T/ipykernel_53377/1577738339.py:12: DtypeWarning: Columns (27580) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)


,Sample ID,cg00000292,cg00002426,cg00003994,cg00005847,cg00006414,cg00007981,cg00008493,cg00008713,cg00009407,...,cg27657283,cg27661264,cg27662379,cg27662877,cg27665659,GSE_number,Age,Age_group,Dataset_ID,Age_group (2 groups)
0,0,0.843562,0.852290,0.092667,0.206619,0.088760,0.026743,0.946611,0.041117,0.066368,...,0.071733,0.387695,0.024861,0.043252,0.043007,GSE19711,69,60-80,4,Old
1,1,0.819140,0.792084,0.077824,0.129618,0.095203,0.057627,0.943816,0.029682,0.049390,...,0.054438,0.217040,0.022823,0.047723,0.051703,GSE19711,75,60-80,4,Old
2,2,0.853076,0.885732,0.057558,0.116209,0.064679,0.028452,0.951909,0.029134,0.076195,...,0.038932,0.417911,0.031721,0.036040,0.040623,GSE19711,58,40-60,4,Young
3,3,0.799517,0.854328,0.063802,0.168209,0.086761,0.029277,0.961985,0.037684,0.065395,...,0.051071,0.241700,0.025137,0.032319,0.043402,GSE19711,62,60-80,4,Young
4,4,0.832239,0.878872,0.049117,0.133839,0.119680,0.024076,0.951260,0.035695,0.069592,...,0.048070,0.379598,0.034993,0.040555,0.047433,GSE19711,57,40-60,4,Young


<h5> The dataset contains 27,584 columns (features) across thousands of samples, making it an extremely high-dimensional dataset. This scale introduced several practical challenges: processing time for even basic dataset operations increased substantially, isolating relevant signals and patterns from the large volume of irrelevant features became more difficult, and the risk of overfitting rose considerably given how many features were available relative to the number of samples. </h5>

<h3>Clean the dataset</h3>

In [125]:
df2 = df.copy()
df2.dtypes
df2.columns

df2 = df2.drop(columns=["Age_group (2 groups)"])
df2 = df2.drop(columns=["GSE_number"])
df2 = df2.drop(columns=["Age_group"])
df2 = df2.drop(columns=["Dataset_ID"])

df2["Age"] = df2["Age"].replace("UNKNOWN", float("nan"))
df2 = df2.dropna(subset=["Age"])
df2["Age"] = pd.to_numeric(df2["Age"])

df2.head()

,Sample ID,cg00000292,cg00002426,cg00003994,cg00005847,cg00006414,cg00007981,cg00008493,cg00008713,cg00009407,...,cg27654142,cg27655855,cg27655905,cg27657249,cg27657283,cg27661264,cg27662379,cg27662877,cg27665659,Age
0,0,0.843562,0.852290,0.092667,0.206619,0.088760,0.026743,0.946611,0.041117,0.066368,...,0.072575,0.454191,0.057836,0.187587,0.071733,0.387695,0.024861,0.043252,0.043007,69.0
1,1,0.819140,0.792084,0.077824,0.129618,0.095203,0.057627,0.943816,0.029682,0.049390,...,0.167140,0.835011,0.078327,0.158581,0.054438,0.217040,0.022823,0.047723,0.051703,75.0
2,2,0.853076,0.885732,0.057558,0.116209,0.064679,0.028452,0.951909,0.029134,0.076195,...,0.067795,0.844276,0.076914,0.205880,0.038932,0.417911,0.031721,0.036040,0.040623,58.0
3,3,0.799517,0.854328,0.063802,0.168209,0.086761,0.029277,0.961985,0.037684,0.065395,...,0.073761,0.816040,0.076892,0.167379,0.051071,0.241700,0.025137,0.032319,0.043402,62.0
4,4,0.832239,0.878872,0.049117,0.133839,0.119680,0.024076,0.951260,0.035695,0.069592,...,0.057765,0.834358,0.069090,0.154178,0.048070,0.379598,0.034993,0.040555,0.047433,57.0


In [127]:
df2['Age'].describe()

count    1851.000000
mean       52.708136
std        16.852839
min        16.000000
25%        40.000000
50%        57.000000
75%        66.000000
max        91.000000
Name: Age, dtype: float64

In [129]:
df2['Age'].isnull().sum()

0

<h5> I removed every column apart from Age. Age_group (2 groups) and Age_group were both closely derived of the same information, but both are attributes that can be recomputed directly from the Age column itself, making them redundant. GSE_number and Dataset_ID are useful for tracing data origin, but carry no predictive value for estimating biological age from methylation data. </h5>

<h3>Split the Data</h3>

In [131]:
x = df2.drop(columns=["Age"]) #features
x = x.drop(columns=["Sample ID"])
y = df2["Age"] #target variable we are trying to predict

xTrainVal, xTest, yTrainVal, yTest = train_test_split(x, y, test_size=0.2, random_state=25, shuffle=True) #first split to just get test data
xTrain, xVal, yTrain, yVal = train_test_split(xTrainVal, yTrainVal, test_size=0.25, random_state=25, shuffle=True) #second split to get train and val data, 
#0.25 so that we end up with 0.8 * 0.25 = 0.20 for val 

print("TRAINING SET (x) shape: ", xTrain.shape)
print("TRAINING SET (y) shape: ", yTrain.shape)
print("VALIDATION SET (x) shape: ", xVal.shape)
print("VALIDATION SET (y) shape: ", yVal.shape)
print("TESTING SET (x) shape: ", xTest.shape)
print("TESTING SET (y) shape: ", yTest.shape)

TRAINING SET (x) shape:  (1110, 27578)
TRAINING SET (y) shape:  (1110,)
VALIDATION SET (x) shape:  (370, 27578)
VALIDATION SET (y) shape:  (370,)
TESTING SET (x) shape:  (371, 27578)
TESTING SET (y) shape:  (371,)


<h5> I split the data into 60% training, 20% validation, and 20% test. This allocation prioritizes giving the model as much data as possible to learn from, while still reserving enough held-out data to reliably evaluate generalization and guard against overfitting.</h5>

<h3>Missing Value Imputation</h3>

In [133]:
xTrain = xTrain.replace('UNKNOWN', float('nan'))
xVal = xVal.replace('UNKNOWN', float('nan'))
xTest = xTest.replace('UNKNOWN', float('nan'))

In [135]:
missingValueRatio = xTrain.isnull().sum().sum() / (len(xTrain) * len(xTrain.columns))
print("Missing Value Ratio for the features (x): ", missingValueRatio)

if missingValueRatio > 0:
    # Using Median Imputation
    from sklearn.impute import SimpleImputer
    medianImputer = SimpleImputer(strategy="median")
    xTrain = pd.DataFrame(medianImputer.fit_transform(xTrain), columns=xTrainVal.columns)
    xTrain = pd.DataFrame(xTrain, columns=xTrainVal.columns)
    xVal = pd.DataFrame(medianImputer.transform(xVal), columns=xTrainVal.columns)
    xTest = pd.DataFrame(medianImputer.transform(xTest), columns=xTrainVal.columns)
    
    yTrain = yTrain.reset_index(drop=True)
    yVal = yVal.reset_index(drop=True)
    yTest = yTest.reset_index(drop=True)
    
missingValueRatio = xTrain.isnull().sum().sum() / (len(xTrain) * len(xTrain.columns))
print("Missing Value Ratio for the features (x): ", missingValueRatio)

Missing Value Ratio for the features (x):  0.0016622467706665256
Missing Value Ratio for the features (x):  0.0


<h5> I used median imputation to fill in the missing values. There weren't many missing values in this dataset, so the choice between mean and median likely wouldn't have changed results dramatically; however, median is more robust to outliers, so I went with that as the safer option. Categorical imputation didn't apply here since every feature is numerical, not categorical.</h5>

<h3>Feature Scaling</h3>

In [78]:
#using standardization scaling
# z = (element - mean) / standard deviation

for column in xTrain.columns[:-1]:
    mean = xTrain[column].mean()
    standardDeviation = xTrain[column].std()
    xTrain[column] = (xTrain[column] - mean) / standardDeviation
    
    xVal[column] = (xVal[column] - mean) / standardDeviation
    xTest[column] = (xTest[column] - mean) / standardDeviation

xTrain

,cg00000292,cg00002426,cg00003994,cg00005847,cg00006414,cg00007981,cg00008493,cg00008713,cg00009407,cg00010193,...,cg27653134,cg27654142,cg27655855,cg27655905,cg27657249,cg27657283,cg27661264,cg27662379,cg27662877,cg27665659
0,0.580536,0.948415,0.798676,-0.758717,0.274823,-0.539761,0.284736,0.083932,0.493556,1.024370,...,0.520909,0.029201,0.240980,-0.094576,-0.788908,0.583370,0.115659,-0.239027,-0.158157,0.049247
1,-1.053543,-1.191563,-0.292136,2.034592,0.166705,-0.077575,-0.396372,-0.027837,-0.348094,-1.225465,...,-1.250262,0.449584,-0.788009,-0.046116,-1.405016,0.362305,-0.254087,0.025147,-0.045552,0.041765
2,-0.110060,0.531492,1.520927,1.861330,0.229997,-0.263686,-0.353851,0.008897,-0.350588,-2.636683,...,-0.422296,0.584617,-0.866579,-0.588585,-1.546548,-0.154808,-0.681522,-0.053408,0.049847,0.035873
3,-0.395771,-0.693283,-0.398839,-0.600669,-0.464146,0.490067,0.299429,0.218916,-0.618657,-0.579416,...,-0.372086,0.510027,-0.056188,-0.268920,1.244744,0.079120,-0.974950,0.104198,-0.356737,0.043291
4,-1.541725,-1.678429,-0.604569,-1.061527,-0.658002,-0.409147,0.207857,0.345546,0.239518,-0.868153,...,0.885618,-0.442324,0.687653,-0.244914,1.338253,-0.714663,0.444127,-0.327117,-0.220607,0.029891
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1105,-1.367548,-1.796783,-0.943873,-0.781721,-0.679354,0.152382,0.377298,0.573359,-0.685146,-0.177849,...,1.050233,-0.348592,0.640118,0.072681,0.611804,0.017487,0.403082,-0.243778,0.015288,0.033781
1106,1.187601,0.723266,-0.021580,-0.580583,-0.447764,0.305127,0.158243,-0.190939,0.322408,-1.028787,...,0.702288,-0.398065,0.449288,-0.126169,-0.098669,-0.135951,0.862784,0.085158,-0.247347,0.024600
1107,-1.132323,-1.394468,-0.685513,-0.349618,-1.104748,-0.649066,0.922630,-0.107870,-0.010356,-0.166029,...,-1.358963,-0.728785,0.957369,-0.551790,0.116092,-0.468479,-0.372201,-0.645687,-0.829889,0.057900
1108,1.169540,1.254617,-0.749272,-0.574371,-0.660803,-0.316017,0.503077,0.267186,-0.829217,-1.449988,...,1.230446,-0.312774,0.770974,-0.233106,1.423310,-0.618321,0.673753,-0.414493,-0.414726,0.046780


<h5> I applied standardized scaling so that all features share a consistent scale. Although every CpG value already falls between 0 and 1, the spread and distribution within that range varies significantly across features, so standardization ensures the model treats each feature's contribution fairly.</h5>

<h3>Feature Selection</h3>

In [81]:
correlation = xTrain.corrwith(yTrain)
print("correlation", correlation)

threshold = 0.50
keptFeatures = correlation[correlation.abs() >= threshold].index

xTrainSelected = xTrain[keptFeatures]
xValSelected = xVal[keptFeatures]
xTestSelected = xTest[keptFeatures]

print("Original dataset shape: ", x.shape)
print("Reduced dataset shape: ", xTrainSelected.shape)
xTrainSelected

correlation cg00000292    0.013319
cg00002426    0.077466
cg00003994    0.205969
cg00005847    0.241599
cg00006414    0.223582
                ...   
cg27657283    0.108552
cg27661264   -0.160603
cg27662379    0.089297
cg27662877    0.151539
cg27665659    0.158409
Length: 27578, dtype: float64
Original dataset shape:  (1851, 27578)
Reduced dataset shape:  (1110, 10)


,cg01803059,cg05331214,cg05380910,cg13379236,cg16744741,cg19776090,cg24450631,cg24957240,cg25431974,cg25809905
0,1.626746,-0.790179,-0.085625,-0.319044,-0.705544,-1.272581,-0.942766,1.023189,0.100485,-0.293906
1,-0.178095,-0.567647,-0.001500,-2.706082,0.920503,-0.071925,-0.852093,0.172432,1.412331,1.279764
2,-0.573076,-2.080746,-0.142271,-2.035704,-1.082926,-0.307576,-0.989538,-0.406378,1.446193,0.062302
3,-1.363612,0.523987,1.186852,1.093765,0.056743,0.635347,1.030716,-0.367172,-0.630440,-0.029085
4,-0.473150,1.478609,-0.483821,0.506102,1.002790,0.303780,0.244332,0.320813,-0.294747,1.203674
...,...,...,...,...,...,...,...,...,...,...
1105,-1.864101,1.363634,0.953725,0.616993,1.166836,0.810969,0.548053,-1.864461,0.147598,0.308007
1106,0.079932,1.333122,0.133480,1.561954,1.501367,0.778495,0.250565,0.572727,-2.593863,1.183084
1107,0.694892,-0.103627,-0.858770,0.208341,1.449974,-1.562487,-1.404811,0.389226,0.552019,-0.177536
1108,-0.407826,-0.099250,1.806051,1.808794,0.590812,0.631973,1.437884,-2.007269,-0.855117,1.149950


<h5> I selected a correlation threshold of 0.5 after testing a range of values (0.1 through 0.55). A clear pattern emerged: higher thresholds reduced R² on both training and validation data while retaining fewer features, whereas lower thresholds inflated training R² substantially but produced unrealistically high validation R² values, a clear sign of overfitting. A threshold of 0.5 offered the best balance between model performance and stability.</h5> 

<h3>Linear Regression</h3>

In [84]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model = LinearRegression()
model.fit(xTrainSelected, yTrain)

yTrainPred = model.predict(xTrainSelected)

mse = mean_squared_error(yTrain, yTrainPred)
rmse = np.sqrt(mse)
r2 = r2_score(yTrain, yTrainPred)

print("Training Results:")
print(f"MSE:  {mse}")
print(f"RMSE: {rmse}")
print(f"R²:   {r2}")

Training Results:
MSE:  140.47303036703283
RMSE: 11.852131891226694
R²:   0.5134527245600968


<h5> This linear regression model was trained on 10 features selected at a correlation threshold of 0.5. The resulting R² of 0.51 indicates that roughly half of the variance in age can be explained by these 10 CpG features alone. While not an especially high figure, this is reasonable to expect given the scale and noise inherent to a dataset of this size. The RMSE of 11.85 indicates the model's age predictions were off by an average of roughly 12 years, albeit not a small margin but a solid starting point considering how few features were used.

These results could likely improve further with a larger feature set, since 10 features represent a tiny fraction of the ~20,000 candidates available. It's genuinely notable that a model built on such a small subset of features was able to capture roughly half the variance in the target, which, honestly, is more than I imagined. </h5>

<h3>Validation Set Evaluation</h3>

In [87]:
print("Validation Results:")

print("Original dataset shape: ", x.shape)

for threshold in [0.3, 0.35, 0.4, 0.45, 0.5]:
    keptFeatures = correlation[correlation.abs() >= threshold].index

    xTrainValSelected = xTrain[keptFeatures]
    xValSel = xVal[keptFeatures]
    xTestSel = xTest[keptFeatures]
    
    if len(keptFeatures) == 0:
        print(f"Threshold {threshold} | No features selected")
        continue
        
    model = LinearRegression()
    model.fit(xTrain[keptFeatures], yTrain)
    
    yValPred = model.predict(xVal[keptFeatures])
    newmse = mean_squared_error(yVal, yValPred)
    newrmse = np.sqrt(newmse)
    newr2 = r2_score(yVal, yValPred)
    
    print(f"Threshold {threshold} | Features: {len(keptFeatures)} | MSE: {newmse} | RMSE: {newrmse} | R²: {newr2}")
    print("Reduced dataset shape: ", xTrainValSelected.shape)

threshold = 0.50
keptFeatures = correlation[correlation.abs() >= threshold].index

xTrainValSelected = xTrain[keptFeatures]
xValSel = xVal[keptFeatures]
xTestSel = xTest[keptFeatures]
xTrainValSelected

Validation Results:
Original dataset shape:  (1851, 27578)
Threshold 0.3 | Features: 1073 | MSE: 4.88327190714121e+78 | RMSE: 2.2098126407325148e+39 | R²: -1.743882521450973e+76
Reduced dataset shape:  (1110, 1073)
Threshold 0.35 | Features: 414 | MSE: 1.1960723282714244e+77 | RMSE: 3.4584278628755936e+38 | R²: -4.2713362420254673e+74
Reduced dataset shape:  (1110, 414)
Threshold 0.4 | Features: 152 | MSE: 6.812797336358166e+75 | RMSE: 8.253967128816401e+37 | R²: -2.4329421795433096e+73
Reduced dataset shape:  (1110, 152)
Threshold 0.45 | Features: 43 | MSE: 5.48861669398149e+75 | RMSE: 7.408519888602237e+37 | R²: -1.9600593416846498e+73
Reduced dataset shape:  (1110, 43)
Threshold 0.5 | Features: 10 | MSE: 145.94031811381024 | RMSE: 12.080576067133977 | R²: 0.47882736252820957
Reduced dataset shape:  (1110, 10)


,cg01803059,cg05331214,cg05380910,cg13379236,cg16744741,cg19776090,cg24450631,cg24957240,cg25431974,cg25809905
0,1.626746,-0.790179,-0.085625,-0.319044,-0.705544,-1.272581,-0.942766,1.023189,0.100485,-0.293906
1,-0.178095,-0.567647,-0.001500,-2.706082,0.920503,-0.071925,-0.852093,0.172432,1.412331,1.279764
2,-0.573076,-2.080746,-0.142271,-2.035704,-1.082926,-0.307576,-0.989538,-0.406378,1.446193,0.062302
3,-1.363612,0.523987,1.186852,1.093765,0.056743,0.635347,1.030716,-0.367172,-0.630440,-0.029085
4,-0.473150,1.478609,-0.483821,0.506102,1.002790,0.303780,0.244332,0.320813,-0.294747,1.203674
...,...,...,...,...,...,...,...,...,...,...
1105,-1.864101,1.363634,0.953725,0.616993,1.166836,0.810969,0.548053,-1.864461,0.147598,0.308007
1106,0.079932,1.333122,0.133480,1.561954,1.501367,0.778495,0.250565,0.572727,-2.593863,1.183084
1107,0.694892,-0.103627,-0.858770,0.208341,1.449974,-1.562487,-1.404811,0.389226,0.552019,-0.177536
1108,-0.407826,-0.099250,1.806051,1.808794,0.590812,0.631973,1.437884,-2.007269,-0.855117,1.149950


<h5> Evaluating this model on the validation set produced an R² of 0.47 at the best-performing threshold (0.5) — closely aligned with the training R² of 0.51. This consistency suggests the model generalizes reasonably well rather than simply memorizing the training data. The validation RMSE matched the training RMSE almost exactly (11.85), reinforcing that performance is stable across both sets. </h5>

<h5> The best performing hyperparameters were a correlation threshold of 0.5 because all other thresholds yielded quite unusual R² values. Realstically, while 10 features is a small number, the model was pushed towards overfitting with a higher number of features, so this low feature count worked best. Median imputation and standardized scaling further supported the model's stability by ensuring that all features contributed equally, without any disproprotionately influencing performance. </h5>

<h3>Polynomial Regression</h3>

In [91]:
correlation = abs(xTrainValSelected.corrwith(yTrain))

top_features = correlation[correlation > 0.52].index

xTrainValPolynomial = xTrainValSelected[top_features]
xValPolynomial = xValSel[top_features]
xTestPolynomial = xTestSel[top_features]
print(xTrainValPolynomial.shape)

degrees = [2, 3, 4]
rmse_array = []

for degree in degrees:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    xTrainPoly = poly.fit_transform(xTrainValPolynomial)
    xValPoly = poly.transform(xValPolynomial)
    
    polynomial_model = LinearRegression()
    polynomial_model.fit(xTrainPoly, yTrain)
    
    yValPred = polynomial_model.predict(xValPoly)
    mse = mean_squared_error(yVal, yValPred)
    rmse = np.sqrt(mse)
    r2 = r2_score(yVal, yValPred)
    rmse_array.append(rmse)
    
    print(f"MSE: {mse}")
    print(f"RMSE: {rmse}")
    print(f"r2: {r2}")
    print(f"degree: {degree}")
    print("~~~~~")

min_rmse = min(rmse_array)
index = rmse_array.index(min_rmse)
best_degree = degrees[index]

best_model = PolynomialFeatures(degree=best_degree, include_bias=False)
xTrainPoly = best_model.fit_transform(xTrainValPolynomial)
xValPoly = best_model.transform(xValPolynomial)
polynomial_model = LinearRegression()
polynomial_model.fit(xTrainPoly, yTrain)
yTrainPred = polynomial_model.predict(xTrainPoly)
yValPred = polynomial_model.predict(xValPoly)

train_mse = mean_squared_error(yTrain, yTrainPred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(yTrain, yTrainPred)

print("\n--- Final Training Set Performance ---")
print(f"MSE: {train_mse:.4f}")
print(f"RMSE: {train_rmse:.4f}")
print(f"R2 Score: {train_r2:.4f}")
print(f"Degree: {best_degree}")

(1110, 5)
MSE: 153.88690843599528
RMSE: 12.40511622017284
r2: 0.450449012455742
degree: 2
~~~~~
MSE: 162.88568980121372
RMSE: 12.762667816769882
r2: 0.41831314569351175
degree: 3
~~~~~
MSE: 391.848830530702
RMSE: 19.795171899498676
r2: -0.3993452332936731
degree: 4
~~~~~

--- Final Training Set Performance ---
MSE: 134.3500
RMSE: 11.5909
R2 Score: 0.5347
Degree: 2


<h5>Increasing the polynomial degree lets the model pick up on non-linear relationships between methylation and age, but it also increases model complexity, especially with a high-dimensional dataset like this one, since polynomial expansion multiplies out the number of features at each degree. This introduces a higher overfitting risk than with linear regression, making feature selection even more critical here. I re-applied correlation-based selection (threshold 0.52) before generating the polynomial features, both to keep things computationally reasonable and to cut down on redundant terms that could destabilize the model. The optimal degree was selected by comparing RMSE across degrees 2, 3, and 4 on the validation set. Degree 2 performed best, with a validation RMSE of 12.41 and R² of 0.45. </h5>

<h3>Generalisation Test</h3>

In [94]:
xTestPoly = best_model.transform(xTestPolynomial)
yTestPred = polynomial_model.predict(xTestPoly)
test_mse = mean_squared_error(yTest, yTestPred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(yTest, yTestPred)

print("--- Final Test Set Performance ---")
print(f"Test MSE:  {test_mse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test R2:   {test_r2:.4f}")

--- Final Test Set Performance ---
Test MSE:  141.7104
Test RMSE: 11.9042
Test R2:   0.4810


In [95]:
path = os.path.join(os.path.dirname(biolearn.__file__), "data", "Horvath1.csv")
print(path)

/opt/anaconda3/lib/python3.11/site-packages/biolearn/data/Horvath1.csv


In [96]:
horvath_df = pd.read_csv(path)
horvath_cpgs = set(horvath_df["CpGmarker"])

my_features = set(top_features)  

overlap = my_features & horvath_cpgs

print(f"My selected features: {len(my_features)}")
print(f"Horvath 353 clock CpGs: {len(horvath_cpgs)}")
print(f"Overlap: {len(overlap)}")
print(overlap)

My selected features: 5
Horvath 353 clock CpGs: 353
Overlap: 2
{'cg16744741', 'cg25809905'}


<h5>On the test set, the final polynomial model (degree 2) achieved an R² of 0.48, an MSE of 141.71, and an RMSE of 11.90, so predictions were off by an average of roughly 12 years on unseen data. Compared to training performance (R² 0.53, RMSE 11.59), there's a moderate drop from training to test, and compared to the linear regression baseline (train R² 0.51, validation R² 0.47), the polynomial model's test performance lands close to the linear model's validation performance, with only a marginal RMSE difference (11.90 vs. 11.85). This suggests the added non-linear complexity from degree-2 polynomial expansion didn't produce a meaningful improvement over the simpler linear model, despite the modest boost in training R².

Comparing performance across training, validation, and test sets, the gap between training R² (0.53) and test R² (0.48) points to mild overfitting: the polynomial model fit the training data somewhat more closely than it was able to generalize to unseen data. That said, the gap is moderate rather than severe. A likely contributing factor is the small selected feature set (5–10 CpG sites): with so few features, even a low polynomial degree like 2 introduces enough additional interaction terms to let the model fit training-specific patterns that don't hold up as well on the test set. A larger, more carefully selected feature set would likely narrow this gap and could make higher polynomial degrees more viable without the same overfitting risk.</h5>